# 第九课：推理优化 —— 让AI更快、更便宜

## 学习目标
- 理解 AI 推理的性能指标：延迟、吞吐量、成本
- 通过代码对比不同模型的响应速度
- 计算 AI 使用成本，理解降本策略
- 了解量化、蒸馏等模型优化技术

> 快、好、便宜——AI 工程化中的不可能三角。

## 环境准备

> 请先运行 `00_Environment_Setup.ipynb` 完成环境配置（安装依赖包 + 设置 API Key），
> 然后再回到本 Notebook。

完成后，运行下面的代码加载环境变量：

In [ ]:
# 从 .env 文件加载 API Key（无需每次输入）
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

# ===== 选一个服务商：只改这一行，其他都不用动 =====
#   'openai'     云端  需要 OPENAI_API_KEY      效果最强，支持 Embedding
#   'deepseek'   云端  需要 DEEPSEEK_API_KEY    云端最便宜，无 Embedding
#   'openrouter' 云端  需要 OPENROUTER_API_KEY  可调用多家模型，无 Embedding
#   'ollama'     本地  不需要 Key，免费离线     先跑 `ollama serve` 并 pull 模型
PROVIDER = 'openai'

# 下面四家都兼容 OpenAI 的接口格式，区别只在：地址、Key、模型名。
PROVIDERS = {
    'openai': {
        'base_url': None,                            # None = 用 OpenAI 官方默认地址
        'api_key': os.getenv('OPENAI_API_KEY'),
        'model': 'gpt-5.6-luna',                     # 小模型：便宜、快
        'model_big': 'gpt-5.6-terra',                # 大模型：贵、强
        'embedding_model': 'text-embedding-3-small',
    },
    'deepseek': {
        'base_url': 'https://api.deepseek.com/v1',
        'api_key': os.getenv('DEEPSEEK_API_KEY'),
        'model': 'deepseek-v4-flash',                # 快、便宜
        'model_big': 'deepseek-v4-pro',              # 更强、更慢；V4 两个模型都会先思考再回答
        'embedding_model': None,                     # DeepSeek 目前不提供 Embedding 接口
    },
    'openrouter': {
        'base_url': 'https://openrouter.ai/api/v1',
        'api_key': os.getenv('OPENROUTER_API_KEY'),
        'model': 'openai/gpt-5.6-luna',
        'model_big': 'openai/gpt-5.6-terra',
        'embedding_model': None,                     # OpenRouter 不转发 Embedding 接口
    },
    'ollama': {
        'base_url': os.getenv('OLLAMA_BASE_URL', 'http://localhost:11434/v1'),
        'api_key': 'ollama',                         # 本地模型不校验 Key
        'model': 'gemma4:e2b-mlx',                   # 需先 ollama pull gemma4:e2b-mlx
        'model_big': 'gemma4:e2b-mlx',
        'embedding_model': 'nomic-embed-text',       # 需先 ollama pull nomic-embed-text
    },
}

cfg = PROVIDERS[PROVIDER]

# 先检查 Key：Key 为空时 OpenAI 客户端会直接抛出一长串报错，不容易看懂。
if not cfg['api_key']:
    raise SystemExit(
        f"没读到 '{PROVIDER}' 的 API Key。请在 .env 文件里补上 {PROVIDER.upper()}_API_KEY，\n"
        f"或者把上面的 PROVIDER 改成 'ollama'，用本地模型运行，完全不需要 Key。"
    )

client = OpenAI(api_key=cfg['api_key'], base_url=cfg['base_url'])

# 后面所有代码都只用这三个变量，换服务商不需要改任何一行业务代码
MODEL = cfg['model']
MODEL_BIG = cfg['model_big']
EMBEDDING_MODEL = cfg['embedding_model']

print(f'连接成功！服务商 = {PROVIDER}，默认模型 = {MODEL}')


---

## 活动一：测量不同模型的响应速度

### 活动目标
用代码精确测量不同模型的响应时间（延迟），理解「模型越大不一定越慢」以及「速度和质量需要权衡」。

In [ ]:
import time
# 活动一：响应速度对比

prompt = '请用100字介绍量子计算的基本概念。'
num_runs = 3  # 每个模型测试3次取平均

# 模型列表
models_to_test = list(dict.fromkeys([MODEL, MODEL_BIG]))  # 没有权限的模型会自动跳过

results = {}

for model_name in models_to_test:
    times = []
    token_counts = []
    print(f'\n测试模型: {model_name}')
    for i in range(num_runs):
        start = time.time()
        try:
            r = client.chat.completions.create(
                model=model_name,
                messages=[{'role':'user','content':prompt}],
                temperature=0.5,
                max_tokens=200)
        except Exception as e:
            print(f'  [跳过] {model_name} 不可用：{str(e)[:100]}')
            break
        elapsed = max(time.time() - start, 1e-6)
        tokens = r.usage.completion_tokens
        times.append(elapsed)
        token_counts.append(tokens)
        print(f'  第{i+1}次: {elapsed:.2f}秒, {tokens} tokens, 速度: {tokens/elapsed:.0f} tok/s')

    if not times:  # 该模型完全没跑通，不进汇总表
        continue
    avg_time = sum(times) / len(times)
    avg_tokens = sum(token_counts) / len(token_counts)
    results[model_name] = {
        'avg_time': avg_time,
        'avg_tokens': avg_tokens,
        'avg_speed': avg_tokens / avg_time
    }

print('\n' + '='*60)
print('汇总对比：')
print(f'{"模型":<20}{"平均耗时(s)":<15}{"平均Token":<15}{"速度(tok/s)":<15}')
print('-'*60)
for model_name, stats in results.items():
    print(f'{model_name:<20}{stats["avg_time"]:<15.2f}{stats["avg_tokens"]:<15.0f}{stats["avg_speed"]:<15.0f}')

print('\n性能指标解释：')
print('  耗时 = 从发送请求到收到完整回答的时间')
print('  Token = AI 生成的文字量（以token计）')
print('  速度 = Token / 耗时（越高越快）')

### 讨论
- 哪个模型最快？和你的直觉一致吗？
- 如果 AI 快50%但准确率降5%，你接受吗？这取决于什么场景？
- TTFT（首字延迟）和完整回答时间，哪个更影响用户体验？

---

## 活动二：计算 AI 使用成本

### 活动目标
用真实的 API 定价信息，计算不同使用场景下的成本。这个练习帮助你建立「AI 也是要花钱的」意识。

> 参考价格（2026年8月）：gpt-5.6-luna 输入 $0.20/1M tokens，输出 $1.20/1M tokens

In [ ]:
# 活动二：AI 使用成本估算

# 定价参考（美元/百万tokens）
# 注意：价格会变，上课前请到各家官网核对（这里只作数量级参考）
pricing = {
    'gpt-5.6-luna': {'input': 0.20, 'output': 1.20},
    'gpt-5.6-terra': {'input': 2.00, 'output': 12.00},
    'deepseek-v4-flash': {'input': 0.14, 'output': 0.28},   # api-docs.deepseek.com，未命中缓存的输入价
    'deepseek-v4-pro': {'input': 0.435, 'output': 0.87},    # 命中缓存时输入价约便宜 50 倍
}

# 场景定义
scenarios = [
    {'name': '个人日常使用', 'users_per_day': 1, 'queries_per_user': 10,
     'input_tokens_per_query': 100, 'output_tokens_per_query': 200},
    {'name': '小型创业公司', 'users_per_day': 500, 'queries_per_user': 5,
     'input_tokens_per_query': 300, 'output_tokens_per_query': 500},
    {'name': '大型企业客服', 'users_per_day': 10000, 'queries_per_user': 3,
     'input_tokens_per_query': 200, 'output_tokens_per_query': 300},
]

print('AI 使用成本估算（月成本，美元）\n')
print(f'{"场景":<20}{"模型":<15}{"日查询量":<12}{"月成本":<12}{"年成本":<12}')
print('-'*75)

for scenario in scenarios:
    daily_queries = scenario['users_per_day'] * scenario['queries_per_user']
    for model_name, price in pricing.items():
        daily_input = daily_queries * scenario['input_tokens_per_query'] / 1_000_000
        daily_output = daily_queries * scenario['output_tokens_per_query'] / 1_000_000
        daily_cost = daily_input * price['input'] + daily_output * price['output']
        monthly_cost = daily_cost * 30
        yearly_cost = monthly_cost * 12
        print(f'{scenario["name"]:<20}{model_name:<20}{daily_queries:<12}${monthly_cost:<11.2f}${yearly_cost:<11.2f}')

print('\n关键发现：')
print('1. gpt-5.6-luna 比 gpt-5.6-terra 便宜约 10 倍')
print('2. 对于简单任务，用便宜的模型能节省大量成本')
print('3. 大型企业客服每月可能需要数千美元——选对模型很重要！')

### 讨论
- 哪个数字最让你惊讶？
- 在你的场景中，模型选择和成本之间如何平衡？
- 什么情况下多花钱用更好的模型是值得的？

---

## 活动三：理解模型「压缩」——量化入门

### 活动目标
量化（Quantization）是降低模型成本和延迟的关键技术。
直观类比：把高清图片（32位权重）压缩成普清图片（8位或4位权重），文件变小但基本能看。

In [ ]:
# 活动三：量化概念演示

# 用数字模拟量化的效果
import random

# 模拟一个模型的权重矩阵（32位浮点数）
print('模拟模型量化效果：\n')

# 生成一些模拟权重
random.seed(42)
weights_32bit = [random.uniform(-1, 1) for _ in range(10)]

# 模拟量化到 4-bit（只有16个可能值）
def quantize_4bit(value):
    """将32位浮点数量化为4位整数（0-15）再映射回原范围"""
    # 映射到 0-15
    normalized = (value + 1) / 2  # 映射到 0-1
    quantized = round(normalized * 15)  # 量化到 0-15
    # 映射回 -1 到 1
    restored = (quantized / 15) * 2 - 1
    return restored

weights_4bit = [quantize_4bit(w) for w in weights_32bit]

print('原始32位权重 vs 量化4位权重：')
print(f'{"原始(32bit)":<15}{"量化(4bit)":<15}{"误差":<10}')
print('-'*40)
total_error = 0
for w32, w4 in zip(weights_32bit, weights_4bit):
    error = abs(w32 - w4)
    total_error += error
    print(f'{w32:<15.6f}{w4:<15.6f}{error:<10.6f}')

print(f'平均误差: {total_error/10:.6f}')
print(f'存储节省: {(1-4/32)*100:.0f}%')

print('实际意义：')
print('  32位模型 = 70GB -> 4位量化模型 = ~9GB')
print('  量化后模型可以在普通笔记本上运行！')
print('  虽然精度有损失，但大多数任务上几乎感觉不到差别。')

### 讨论
- 如果你的手机能本地运行一个「够用」的 AI 模型，你最想用它做什么？
- 量化的代价是精度损失——什么场景下不能接受这种损失？
- 为什么量化后的模型速度更快？（提示：更小的数据 = 更快的内存读写）

---

## 本节回顾

| 技能 | 说明 |
|------|------|
| 速度测量 | 用代码精确测量模型响应时间和吞吐量 |
| 成本估算 | 基于真实定价计算不同场景的 AI 使用成本 |
| 量化理解 | 通过模拟理解量化如何压缩模型 |

### 课后练习
1. 用同样的方法测量你使用的其他 AI 服务的响应速度
2. 估算你自己日常使用 AI 的年成本（如果按 API 计费）
3. 搜索「GGUF quantized models」了解可以在本地运行的量化模型